# Level 1 — Prompt Anatomy and an Agent with a Brain, but No Hands

We will first learn prompt anatomy, then build a simple AI Data Analyst agent for an e-commerce marketplace.

**Teaching narration:**
> “At this stage, our agent has a brain, but no hands. It can understand a question and generate SQL, but it has no tools to access the database. It can prepare a query, but it cannot execute it or retrieve real data.”

**Goal:** Create an agent with `create_agent()`, a model, and an empty tools list. No direct `llm.invoke()` call, database connection, or SQL execution.

## End-to-end agent workflow — Version 1

```text
User Question
     |
     v
AI Data Analyst Agent (model + system prompt)
     |
     v
Generate proposed SQL
     |
     v
Return SQL + explanation + missing information
```

**Tools available:** None (`tools=[]`). No Pinecone retrieval, SQL validation, database execution, or web search occurs in this version.

## 1. What is prompt anatomy?

A prompt is the set of instructions and information we give an LLM to guide its response. A well-structured prompt often contains these components:

| Component | What it does | Example for our SQL agent |
|---|---|---|
| **Role** | Defines who the model should act as | You are an AI Data Analyst for an e-commerce marketplace. |
| **Context** | Provides useful background | The marketplace stores orders, products, customers, payments, and inventory. |
| **Task** | States what the model must do | Convert a business question into SQL. |
| **Input** | Supplies the specific request or data | How many orders were cancelled? |
| **Constraints** | Sets boundaries and rules | Do not invent column names. Do not execute SQL. |
| **Output format** | Specifies how the answer should look | Return SQL and a short explanation. |
| **Examples** | Demonstrates the expected pattern | Show a sample question and the SQL it should produce. |
| **Evaluation criteria** | Describes what a good answer must satisfy | SQL should address the question and use known schema details. |

Not every prompt needs every component. Choose components based on the task and how much guidance the model needs.

### Connect prompt anatomy to the agent

The system prompt gives the agent its role, context, task, constraints, and output format. The model is its **brain**. Tools are its **hands**, allowing it to perform actions such as querying a database.

In this lesson, we deliberately give the agent no tools.

## 2. Anatomy of our Text-to-SQL prompt

We will use these components:

1. **Role:** AI Data Analyst Agent.
2. **Context:** E-commerce marketplace database and its known tables.
3. **Task:** Translate a natural-language business question into SQL.
4. **Constraints:** Generate read-only SQL only; do not execute queries or invent schema details.
5. **Output format:** Show the question, SQL, explanation, and assumptions.
6. **Fallback:** If required columns or relationships are unknown, state what schema information is missing instead of guessing.

**Important limitation:** A list of table names is not a full database schema. Without column names and relationships, the model cannot reliably guarantee executable SQL. We make that limitation explicit in the prompt.

## 3. Install dependencies

This notebook uses LangChain's OpenAI integration and `python-dotenv` to load the API key.

In [ ]:
%pip install -q langchain-openai python-dotenv

## 4. Configure the API key and model

Set `OPENAI_API_KEY` in your environment or in a `.env` file in the notebook's working directory. The model ID below is `gpt-6-astra`; API access and availability depend on your account.

In [ ]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent

load_dotenv()

if not os.environ.get("OPENAI_API_KEY"):
    raise ValueError("Please set OPENAI_API_KEY in your environment or .env file.")

MODEL = "gpt-6-astra"
model = ChatOpenAI(model=MODEL)

## 5. Build the system prompt from its components

Instead of writing one long prompt without structure, we make each component visible. This helps us teach, inspect, and improve the prompt later.

In [ ]:
SYSTEM_PROMPT = """
ROLE
You are an AI Data Analyst Agent for an e-commerce marketplace.

CONTEXT
The database is described by these known table names:
orders, order_items, products, customers, shipments, payment_transactions,
reviews, suppliers, warehouses, inventory, promotions, categories,
customer_addresses, product_suppliers
Only table names are currently known. Column names, data types, keys, and
relationships have not been provided in this version.

TASK
Translate the user's natural-language business question into a proposed SQL query.
Identify the metric, filters, joins, aggregation, and date boundaries required.

CAPABILITIES
You have no tools in this version. You cannot inspect schema metadata, execute SQL,
validate a query against PostgreSQL, or retrieve real database results.

CONSTRAINTS
- Propose read-only SELECT SQL only.
- Never claim that SQL was executed or that results were retrieved.
- Never invent column names, relationships, values, or results.
- If the available schema is insufficient, state what information is missing instead
  of guessing.
- Consider table grain, join cardinality, NULLs, and double-counting.
- For percentages, identify the intended numerator and denominator.
- Do not expose private chain-of-thought; provide a concise explanation only.

OUTPUT FORMAT
Return these sections:
QUESTION: Restate the business question briefly.
PROPOSED SQL: Provide SQL only if it can be drafted responsibly; otherwise write UNKNOWN.
EXPLANATION: Briefly explain the query's intent.
ASSUMPTIONS / MISSING INFORMATION: List assumptions and schema details still needed.
"""


## 6. Create the agent — brain, but no hands

We use LangChain's `create_agent()` to create an agent. We do **not** call the model directly with `llm.invoke()`.

- `model=model` gives the agent its brain.
- `tools=[]` means it has no hands: it cannot query a database or use an external tool.
- `system_prompt=SYSTEM_PROMPT` gives it instructions and a response format.

The agent can generate SQL text, but it cannot execute that SQL.

In [ ]:
agent = create_agent(
    model=model,
    tools=[],
    system_prompt=SYSTEM_PROMPT,
)


def ask_agent(question: str) -> str:
    result = agent.invoke(
        {
            "messages": [
                {"role": "user", "content": question}
            ]
        }
    )
    return result["messages"][-1].content


question = "How many orders were cancelled?"

print("QUESTION:")
print(question)
print("\nAGENT RESPONSE:")
print(ask_agent(question))

## 7. Try a few more business questions

The agent can respond to different questions. It can propose SQL, but it cannot run the query or verify results because no tools are connected.

In [ ]:
questions = [
    "How many orders were cancelled?",
    "What are the top 10 best-selling products?",
    "What percentage of orders were cancelled?",
    "Which products have inventory below 100 units?",
    "What is the average order value?",
]

for question in questions:
    print("\n" + "=" * 80)
    print("QUESTION:", question)
    print("=" * 80)
    print(ask_agent(question))

## 8. Key learning points

- **Prompt anatomy** gives the agent clear role, context, task, constraints, and output instructions.
- `create_agent()` creates the agent workflow; we do not call the model directly.
- The model is the agent's **brain**.
- Tools are the agent's **hands**. With `tools=[]`, it cannot execute SQL or access the database.
- The agent can generate a query, but table names alone are not enough to guarantee valid SQL. Add the actual schema before relying on generated queries.
- The agent must not claim it queried the database or report invented results.

**Teaching narration:**
> “We have built an agent with a brain, but no hands. It can understand the request and prepare a SQL query. In the next lesson, we will give it hands by connecting a `run_query` tool. Then it can execute SQL, observe database results, and answer using actual data.”